# पाठ 01 - एआई एजेंट्स का परिचय

**शुरुआती लोगों के लिए एआई एजेंट्स** कोर्स में आपका स्वागत है!

एक **एआई एजेंट** एक प्रोग्राम होता है जो एक बड़े भाषा मॉडल (LLM) को अपनी तर्कशक्ति इंजन के रूप में उपयोग करता है और वास्तविक दुनिया में *कार्रवाइयां* कर सकता है — APIs कॉल करना, डेटाबेस क्वेरी करना, या कोड चलाना — ताकि उपयोगकर्ता की ओर से किसी लक्ष्य को पूरा किया जा सके।

इस नोटबुक में आप अपना पहला एजेंट बनाएंगे: एक **ट्रैवल एजेंट** जो छुट्टियों के लिए गंतव्य सुझाता है। इस मार्ग में आप सीखेंगे कि कैसे:

1. **माइक्रोसॉफ्ट एजेंट फ्रेमवर्क** का उपयोग करके Azure AI Foundry Agent सेवा से कनेक्ट करें।
2. एजेंट को एक **टूल** दें — एक साधारण पायथन फ़ंक्शन जिसे वह कॉल कर सकता है।
3. एजेंट चलाएं और उसकी प्रतिक्रिया देखें।
4. एजेंट की प्रतिक्रिया को टोकन-बाय-टोकन स्ट्रीम करें।


## सेटअप

इस नोटबुक को चलाने से पहले, सुनिश्चित करें कि आपके पास है:

1. **एक Azure AI Foundry प्रोजेक्ट** जिसमें एक तैनात चैट मॉडल हो (जैसे `gpt-4o-mini`)।
2. **Azure CLI के साथ लॉग इन किया हुआ हो** — अपने टर्मिनल में `az login` चलाएं।
3. **आवश्यक पर्यावरण चर सेट किए गए हों:**
   - `AZURE_AI_PROJECT_ENDPOINT` — आपके Azure AI Foundry प्रोजेक्ट का एंडपॉइंट।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — आपके तैनात मॉडल का नाम।

नीचे वाला सेल उन Python पैकेजों को इंस्टॉल करता है जिनकी आपको जरूरत है।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## अपना पहला एजेंट बनाना

एक एजेंट को दो चीजों की आवश्यकता होती है:

- **निर्देश** जो उसे बताएं *वह कौन है* और *कैसे व्यवहार करना है* (एक सिस्टम प्रॉम्प्ट)।
- **उपकरण** — Python फ़ंक्शन जिन्हें `@tool` से सजाया जाता है जिन्हें एजेंट जानकारी प्राप्त करने या क्रियाएं करने के लिए कॉल कर सकता है।

नीचे हमने एक सरल उपकरण परिभाषित किया है जो लोकप्रिय छुट्टी स्थानों की सूची लौटाता है। जब कोई उपयोगकर्ता यात्रा सिफारिशें मांगेगा, तो एजेंट इस उपकरण का उपयोग करेगा।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रीमिंग प्रतिक्रियाएँ

एक अधिक इंटरैक्टिव अनुभव के लिए आप एजेंट की प्रतिक्रिया को **स्ट्रीम** कर सकते हैं। पूरी प्रतिक्रिया का इंतजार करने के बजाय, एजेंट जैसे-जैसे टेक्स्ट उत्पन्न करता है, उसे टुकड़ों में देता है। यह विशेष रूप से चैट इंटरफेस में उपयोगी होता है जहाँ आप आउटपुट को वास्तविक समय में प्रदर्शित करना चाहते हैं।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

इस पाठ में आपने सीखा कि कैसे:

- **एक प्रदाता बनाएँ** जो `AzureAIProjectAgentProvider` के माध्यम से Azure AI Foundry Agent Service से जुड़ता है।
- **एक उपकरण परिभाषित करें** `@tool` डेकोरेटर का उपयोग करके ताकि एजेंट आपके Python फंक्शंस को कॉल कर सके।
- **एजेंट चलाएँ** एक उपयोगकर्ता संदेश के साथ और इसकी प्रतिक्रिया प्रिंट करें।
- **रियल-टाइम आउटपुट के लिए प्रतिक्रिया स्ट्रीम करें**।

अगले पाठ में हम एजेंटिक फ्रेमवर्क्स को और गहराई से समझेंगे और सीखेंगे कि कैसे एजेंट्स को अधिक शक्तिशाली उपकरण और मल्टी-स्टेप तर्क क्षमताएँ दी जा सकती हैं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
